# 25 — How the switchers saw Trump before they voted for him (ANES 2016–2020–2024 panel)

**Question.** Did the Biden voters who switched to Trump in 2024 already see him as someone who "really cares about people like you" — a champion rather than an ordinary Republican — while they were voting against him?

**Plan, fixed before results.** Same panel, weight and groups as notebook 20 (Biden 2020 voters by their 2020 report: 856 loyal, 21 switchers, 64 stay-home; Trump→Trump for reference). ANES asks in every wave, before the election, how well four phrases describe each candidate (1 extremely well … 5 not at all, rescaled 0–1 with 1 = describes well): strong leader, *really cares about people like you*, knowledgeable, honest. The Republican candidate is Trump in all three waves. Primary measure: Trump "cares" by group and wave, and the switcher − loyal gap with bootstrap 95% intervals. Supporting the hypothesis: switchers rating Trump as caring in 2016/2020, and more on "cares" than on honesty or knowledge. Secondary: the Democratic candidate's traits, Trump's minus the Democrat's, and candidate thermometers. With 21 switchers (effective n ≈ 11), weighted and unweighted figures are both reported.

In [1]:
import numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
p = pd.read_csv('../data/raw/anes/anes_panel_2016_2024.csv'); p = p[p.weight > 0].copy()
p['grp'] = np.select([(p.vote20 == 'D') & (p.vote24 == 'D'), (p.vote20 == 'D') & (p.vote24 == 'R'), (p.vote20 == 'D') & (p.vote24 == 'NV'),
                      (p.vote20 == 'R') & (p.vote24 == 'R')], ['loyal', 'switch', 'stayhome', 'TT'], None)
for t in ['cares', 'honest', 'leader', 'knows']:
    for w in ('16', '20', '24'):
        p[f'net_{t}_{w}'] = p[f'trump_{t}_{w}'] - p[f'dem_{t}_{w}']   # Trump minus the Democratic candidate
def wm(x, w):
    m = x.notna(); return np.average(x[m], weights=w[m]) if m.any() else np.nan
G = ['loyal', 'switch', 'stayhome', 'TT']
rows = []
rng = np.random.default_rng(25)
b = p[p.grp.isin(['loyal', 'switch'])]
for v in ['trump_cares', 'trump_honest', 'trump_leader', 'trump_knows', 'dem_cares', 'dem_honest', 'net_cares', 'net_honest', 'therm_trump', 'therm_dem_cand']:
    for w in ('16', '20', '24'):
        c = f'{v}_{w}'
        g = lambda d: wm(d.loc[d.grp == 'switch', c], d.loc[d.grp == 'switch', 'weight']) - wm(d.loc[d.grp == 'loyal', c], d.loc[d.grp == 'loyal', 'weight'])
        bs = [g(b.groupby('grp', group_keys=False).apply(lambda s: s.sample(len(s), replace=True, random_state=rng.integers(1e9)))) for _ in range(1000)]
        rows.append({'item': v, 'wave': 2000 + int(w), **{gg: wm(p.loc[p.grp == gg, c], p.loc[p.grp == gg, 'weight']) for gg in G},
                     'switch_unwtd': p.loc[p.grp == 'switch', c].mean(), 'gap': g(b), 'lo': np.percentile(bs, 2.5), 'hi': np.percentile(bs, 97.5)})
t = pd.DataFrame(rows); t.to_csv('../outputs/tables/anes_panel_candidate_traits.csv', index=False)
print(t.round(2).to_string(index=False))

          item  wave  loyal  switch  stayhome    TT  switch_unwtd    gap     lo     hi
   trump_cares  2016   0.06    0.16      0.06  0.53          0.24   0.10   0.01   0.22
   trump_cares  2020   0.02    0.16      0.03  0.75          0.10   0.14  -0.00   0.33
   trump_cares  2024   0.02    0.47      0.08  0.77          0.39   0.46   0.28   0.65
  trump_honest  2016   0.11    0.19      0.12  0.56          0.31   0.08  -0.02   0.22
  trump_honest  2020   0.01    0.14      0.06  0.63          0.10   0.13  -0.00   0.30
  trump_honest  2024   0.01    0.37      0.08  0.61          0.27   0.36   0.12   0.59
  trump_leader  2016   0.14    0.33      0.19  0.66          0.37   0.18  -0.00   0.39
  trump_leader  2020   0.04    0.18      0.06  0.82          0.18   0.14  -0.00   0.34
  trump_leader  2024   0.08    0.57      0.24  0.85          0.57   0.50   0.25   0.70
   trump_knows  2016   0.12    0.27      0.16  0.56          0.27   0.15  -0.00   0.30
   trump_knows  2020   0.03    0.18      0.

In [2]:
# Share saying 'really cares about people like you' describes Trump at least moderately well (codes 1-3 -> >= 0.5)
rows = []
for w in ('16', '20', '24'):
    c = f'trump_cares_{w}'
    rows.append({'wave': 2000 + int(w), **{g: wm((p.loc[p.grp == g, c] >= 0.5).astype(float).where(p.loc[p.grp == g, c].notna()), p.loc[p.grp == g, 'weight']) for g in G},
                 'switch_N': int(p.loc[p.grp == 'switch', c].notna().sum()), 'switch_count': int((p.loc[p.grp == 'switch', c] >= 0.5).sum())})
share = pd.DataFrame(rows).set_index('wave'); print((100 * share[G]).round(0)); print(share[['switch_count', 'switch_N']])
share.to_csv('../outputs/tables/anes_panel_trump_cares_share.csv')

      loyal  switch  stayhome    TT
wave                               
2016    5.0    17.0       7.0  72.0
2020    1.0    17.0       4.0  91.0
2024    1.0    55.0       7.0  92.0
      switch_count  switch_N
wave                        
2016             7        21
2020             2        21
2024            10        20
